### Additional Dataset (Vital Signs) -- 68 more 12th hour(10/29/2024)

In [2]:
# 68 12th hour file rename
import os
import pandas as pd
import shutil
import re

# Define paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/02b_Vital_Raw_68_12th"
save_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/02b_Vital_Raw_68_12th_RENAME"

# Ensure the save path exists
os.makedirs(save_path, exist_ok=True)

# Load the df_new_file from CSV
df_new_file = pd.read_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/Study23_Tag141_EventList.csv")  # Adjust the path

# Ensure time columns are in datetime format
df_new_file['Time Start'] = pd.to_datetime(df_new_file['Time Start'])
df_new_file['Time Stop'] = pd.to_datetime(df_new_file['Time Stop'])

# Create new columns for the original and renamed file names
df_new_file['Original_FileName'] = ""
df_new_file['Renamed_FileName'] = ""

# Initialize counters for debugging
files_processed = 0
files_skipped = 0

# Track generated file names to identify duplicates
generated_file_names = {}

# Get a list of all CSV files in the raw_data_path
all_files = [f for f in os.listdir(raw_data_path) if f.endswith('.csv')]

# Function to extract the numeric part from the file name (assuming the files have numbers in their names)
def extract_file_number(file_name):
    match = re.search(r'\d+', file_name)  # Extract the first numeric part of the file name
    return int(match.group()) if match else float('inf')  # Convert to int for sorting; if no number, treat it as infinity

# Sort the files based on the numeric value extracted from the file names
all_files = sorted(all_files, key=extract_file_number)
# print("Sorted files:", all_files)

# Loop through each row in df_new_file
for index, row in df_new_file.iterrows():
    if index < len(all_files):
        # Get the file name from the directory
        file_name = all_files[index]
        file_path = os.path.join(raw_data_path, file_name)

        # Read the CSV file
        df = pd.read_csv(file_path)

        # Ensure the Time column is in datetime format
        df['Time'] = pd.to_datetime(df['Time'])

        # Filter the data based on the 1st_Time_Start and 1st_Time_Stop
        start_time = row['Time Start']
        stop_time = row['Time Stop']
        filtered_df = df[(df['Time'] >= start_time) & (df['Time'] <= stop_time)]

        # If data is found within the time range
        if not filtered_df.empty:
            # Generate the new file name using PID and 1st_Time_Start
            pid = row['Patient ID']
            new_file_name = f"{pid}_{start_time.strftime('%Y%m%d_%H')}_Vital_12th.csv"

            # Track file names to identify duplicates
            if new_file_name in generated_file_names:
                print(f"Duplicate file name found: {new_file_name} for row {index}")
            else:
                generated_file_names[new_file_name] = file_path

                # Copy the original file to the new path with the new name
                new_file_path = os.path.join(save_path, new_file_name)
                shutil.copy(file_path, new_file_path)

                # Update df_new_file with the original and new file names
                df_new_file.at[index, 'Original_FileName'] = file_name
                df_new_file.at[index, 'Renamed_FileName'] = new_file_name

                files_processed += 1
                # print(f"Processed and renamed: {new_file_name}")
        else:
            print(f"No data found for time range: {start_time} to {stop_time} in file: {file_name}")
            files_skipped += 1
    else:
        print(f"No more files to process in the directory for row {index}")
        files_skipped += 1

# Debugging info
print(f"Total files processed: {files_processed}")
print(f"Total files skipped (due to missing files or no data in range): {files_skipped}")

# Check for missing files in df_new_file
missing_files = df_new_file[df_new_file['Renamed_FileName'] == ""]
if not missing_files.empty:
    print(f"Missing files or no data in time range for the following rows:\n{missing_files[['Patient ID', 'Time Start', 'Time Stop']]}")

# Save the updated df_new_file
df_new_file.to_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/Study23_Tag141_EventList_v2.csv", index=False)  # Adjust the path

In [6]:
# Update df_combine file
import os
import pandas as pd
from datetime import datetime

# File paths
df_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
vital_data_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/02b_Vital_Raw_68_12th_RENAME"

# Load sheet2 from the df_combined Excel file
df = pd.read_excel(df_path, sheet_name='Sheet2')

# Ensure the 12th_Time_Start column is in datetime format
df['12th_Time_Start'] = pd.to_datetime(df['12th_Time_Start'], errors='coerce')

# Loop through each file in the VitalData directory
for file_name in os.listdir(vital_data_dir):
    if file_name.endswith(".csv"):  # assuming the files are Excel sheets
        # Extract pid, date, and hour parts from the file name
        parts = file_name.split('_')
        if len(parts) < 3:
            continue  # skip files that don't match the expected pattern

        pid, date_str, hour_str = parts[0], parts[1], parts[2]
        file_date = datetime.strptime(date_str, "%Y%m%d").date()
        file_hour = int(hour_str)

        # Find rows in df where pid, date, and hour match
        matching_rows = df[
            (df['PID'] == int(pid)) &
            (df['12th_Time_Start'].dt.date == file_date) &
            (df['12th_Time_Start'].dt.hour == file_hour)
        ]

        # Append the file name to the matched rows in the FileName_Vital_12th column
        for index in matching_rows.index:
            existing_value = df.at[index, 'FileName_Vital_12th']
            if pd.isna(existing_value):  # If the cell is NaN, initialize it as an empty string
                existing_value = ""
            # Append the file name, separated by a comma if there’s already content
            df.at[index, 'FileName_Vital_12th'] = f"{existing_value}, {file_name}" if existing_value else file_name

# Save the updated DataFrame to a new sheet in the Excel file
with pd.ExcelWriter(df_path, engine="openpyxl", mode="a") as writer:
    df.to_excel(writer, sheet_name="Sheet3", index=False)

print("File names added to a new sheet 'Sheet3' in df_combine successfully.")


File names added to a new sheet 'Sheet3' in df_combine successfully.


In [9]:
import os
import pandas as pd

# File paths
df_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
vital_data_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/02_Vital_Raw_226_12th"

# Load Sheet3 from the df_combined Excel file
df = pd.read_excel(df_path, sheet_name='Sheet3')

# Get the list of CSV file names in the VitalData directory
csv_files = {f.strip() for f in os.listdir(vital_data_dir) if f.endswith(".csv")}

# Extract and split file names in the FileName_Vital_12th column
sheet_file_names = set()
for file_name_entry in df['FileName_Vital_12th'].dropna():  # Drop NaN entries
    file_names = [name.strip() for name in file_name_entry.split(',')]  # Split and strip whitespace
    sheet_file_names.update(file_names)

# Find files in the directory but not in the DataFrame
missing_files = csv_files - sheet_file_names

# Print missing file names
print("Files in directory but not in Sheet3 FileName_Vital_12th column:")
for missing_file in missing_files:
    print(missing_file)

# Optionally, check if the known missing file is in the results
known_missing_file = "2951_20231005_05_Vital_12th.csv"
print(f"\nIs '{known_missing_file}' missing? {'Yes' if known_missing_file in missing_files else 'No'}")

Files in directory but not in Sheet3 FileName_Vital_12th column:
132254_20231008_19_Vital_12th.csv

Is '2951_20231005_05_Vital_12th.csv' missing? No


### Additional Dataset (Vital Signs)

In [1]:
import pandas as pd

# Path to your Excel file
file_path = '/home/liuwent/2-PARDS/FOCI_screening_PARDS_20240919.xlsx'

# Read the specific sheet
df1 = pd.read_excel(file_path, sheet_name='TRAPPARDS')
df2 = pd.read_excel(file_path, sheet_name='TRAPPARDS (2)')

# Display the first few rows of the dataframe
print(df1.head())
print(df2.head())

                              PatientID        MRN               DOB
0  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00
1  D7970BC6-445C-E311-800F-00215A9B0094  100205702  03/16/2009 00:00
2  1912AB08-455C-E311-800F-00215A9B0094  100151965  04/12/2000 00:00
3  1E4D6298-455C-E311-800F-00215A9B0094   41994058  09/11/2007 00:00
4  BF94D928-885E-E311-800F-00215A9B0094  100229346  06/02/2010 00:00
                              PatientID                           EncounterID  \
0  43C395B6-395C-E311-800F-00215A9B0094  A0C91FE8-140E-EE11-A19F-0025B5000031   
1  43C395B6-395C-E311-800F-00215A9B0094  E4E6A583-180E-EE11-A19F-0025B5000031   
2  43C395B6-395C-E311-800F-00215A9B0094  473D1FFF-180E-EE11-A19F-0025B5000031   
3  43C395B6-395C-E311-800F-00215A9B0094  87B96123-1A0E-EE11-A19F-0025B5000031   
4  43C395B6-395C-E311-800F-00215A9B0094  F7610F91-1E0E-EE11-A19F-0025B5000031   

          AdmitDate     DischargeDate  ICUStayYN  ICUDays  ICUMinutes  \
0  06/18/2023 16:20  06/21

In [2]:
df1.shape

(454, 3)

In [3]:
len(df1['PatientID'].unique())

454

In [4]:
len(df1['MRN'].unique())

454

In [5]:
df2.shape

(39131, 10)

In [6]:
len(df2['PatientID'].unique())

454

In [7]:
# Merge the dataframes on the "PatientID" column
merged_df = pd.merge(df1, df2, on='PatientID')

# Format the 'MRN' column to have exactly 9 digits with leading zeros
merged_df['MRN'] = merged_df['MRN'].astype(str).str.zfill(9)

# Display the first few rows of the merged dataframe
print(merged_df.head())


                              PatientID        MRN               DOB  \
0  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00   
1  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00   
2  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00   
3  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00   
4  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00   

                            EncounterID         AdmitDate     DischargeDate  \
0  A0C91FE8-140E-EE11-A19F-0025B5000031  06/18/2023 16:20  06/21/2023 12:35   
1  E4E6A583-180E-EE11-A19F-0025B5000031  06/18/2023 16:45  06/18/2023 16:45   
2  473D1FFF-180E-EE11-A19F-0025B5000031  06/18/2023 16:50  06/18/2023 16:50   
3  87B96123-1A0E-EE11-A19F-0025B5000031  06/18/2023 16:55  06/18/2023 16:55   
4  F7610F91-1E0E-EE11-A19F-0025B5000031  06/18/2023 17:25  06/18/2023 17:25   

   ICUStayYN  ICUDays  ICUMinutes HospitalAdmissionDate HospitalDischargeDate  \
0        1.

In [8]:
merged_df.shape

(39131, 12)

In [9]:
len(merged_df['MRN'].unique())

454

In [10]:
len(merged_df['PatientID'].unique())

454

In [11]:
# Keep only the unique combinations of 'PatientID' and 'HospitalAdmissionDate'
merged_df_sub = merged_df.drop_duplicates(subset=['PatientID', 'HospitalAdmissionDate'])

# Group by 'PatientID' and fill NaNs in 'HospitalAdmissionDate' with the corresponding 'AdmitDate'
def fill_hospital_admission_date(group):
    if group['HospitalAdmissionDate'].isna().all():
        group['HospitalAdmissionDate'] = group['AdmitDate']
    return group

merged_df_sub = merged_df_sub.groupby('PatientID').apply(fill_hospital_admission_date).reset_index(drop=True)
merged_df_sub.shape

(1464, 12)

In [12]:
len(merged_df_sub['MRN'].unique())

454

In [13]:
len(merged_df_sub['PatientID'].unique())

454

In [14]:
# Drop rows where 'HospitalAdmissionDate' is duplicated
merged_df_sub_sub = merged_df_sub.drop_duplicates(subset=['PatientID', 'HospitalAdmissionDate'])

# Drop rows where 'HospitalAdmissionDate' is NaN
merged_df_sub_sub = merged_df_sub_sub.dropna(subset=['HospitalAdmissionDate'])

merged_df_sub_sub.shape

(1011, 12)

In [15]:
len(merged_df_sub_sub['MRN'].unique())

454

In [16]:
len(merged_df_sub_sub['PatientID'].unique())

454

In [17]:
df = merged_df_sub_sub
df.shape

(1011, 12)

In [18]:
import pandas as pd

# Convert 'HospitalAdmissionDate' and 'DOB' to datetime if they are not already
df['HospitalAdmissionDate'] = pd.to_datetime(df['HospitalAdmissionDate'])
df['DOB'] = pd.to_datetime(df['DOB'])

# Calculate the age in years
df['Age'] = (df['HospitalAdmissionDate'] - df['DOB']).dt.total_seconds() / (365.25 * 24 * 60 * 60)

# Drop the specified columns
df_sub = df.drop(columns=['PatientID', 'EncounterID', 'ICUMinutes', 'AdmitDate', 'DischargeDate'])

# Reorder and rename the columns
new_column_order = ['MRN', 'DOB', 'Age', 'Weight', 'HospitalAdmissionDate', 
                    'HospitalDischargeDate', 'ICUStayYN', 'ICUDays']
df_sub = df_sub[new_column_order]

df_sub


,MRN,DOB,Age,Weight,HospitalAdmissionDate,HospitalDischargeDate,ICUStayYN,ICUDays
0,100117962,2010-09-02,12.793102,1329.81,2023-06-18 16:20:00,06/21/2023 12:35,1.0,1.0
3,100205702,2009-03-16,14.804356,1477.96,2024-01-04 06:59:00,02/19/2024 15:19,1.0,12.0
5,100151965,2000-04-12,24.211834,2804.25,2024-06-28 08:56:00,07/13/2024 18:14,1.0,1.0
6,100151965,2000-04-12,24.242300,NaN,2024-07-09 12:00:00,07/09/2024 23:59,0.0,NaN
7,100151965,2000-04-12,24.222741,NaN,2024-07-02 08:33:00,07/02/2024 23:59,0.0,NaN
...,...,...,...,...,...,...,...,...
1459,100574112,2015-06-03,8.497220,NaN,2023-12-01 14:38:00,12/01/2023 23:59,0.0,NaN
1460,100574112,2015-06-03,8.496378,NaN,2023-12-01 07:15:00,12/01/2023 14:37,0.0,NaN
1461,100574112,2015-06-03,8.432406,793.66,2023-11-07 22:28:00,02/14/2024 16:36,1.0,99.0
1462,100574112,2015-06-03,8.582181,NaN,2024-01-01 15:24:00,01/01/2024 23:59,0.0,NaN


In [19]:
len(df_sub['MRN'].unique())

454

In [21]:
import pandas as pd

df_sub.to_csv('DOB.csv', index=False)

print("DataFrame saved to 'DOB.csv'")


DataFrame saved to 'DOB.csv'


In [20]:
# Ensure 'HospitalAdmissionDate' is in datetime format
df_sub['HospitalAdmissionDate_Date'] = pd.to_datetime(df_sub['HospitalAdmissionDate'])

# Check the minimum date
min_date = df_sub['HospitalAdmissionDate_Date'].min()

# Check the maximum date
max_date = df_sub['HospitalAdmissionDate_Date'].max()

# Display the results
print(f"Minimum Hospital Admission Date: {min_date}")
print(f"Maximum Hospital Admission Date: {max_date}")

Minimum Hospital Admission Date: 2022-08-11 01:12:00
Maximum Hospital Admission Date: 2024-07-21 01:50:00


### Identify the starting points of each patient's timeline.

In [ ]:
sample_events_MRN = df_sub[['MRN', 'HospitalAdmissionDate']]

# Rename the MRN and HospitalAdmissionDate columns to mrn and Time Start
sample_events_MRN = sample_events_MRN.copy()
sample_events_MRN.rename(columns={'MRN': 'mrn', 'HospitalAdmissionDate': 'Time Start'}, inplace=True)

# Format the 'mrn' column to have exactly 9 digits with leading zeros
sample_events_MRN['mrn'] = sample_events_MRN['mrn'].astype(str).str.zfill(9)

sample_events_MRN

In [ ]:
len(sample_events_MRN['mrn'].unique())

#### 2 hours before hospital admission date.

In [ ]:
import pandas as pd

result_df = sample_events_MRN.copy()

result_df['Time Stop'] = result_df['Time Start']

# Convert 'Time Start' and 'Time Stop' to datetime format
result_df['Time Start'] = pd.to_datetime(result_df['Time Start'])
result_df['Time Stop'] = pd.to_datetime(result_df['Time Stop'])

# Create new 'Time Start' as 'Time Stop' minus 6 hours
result_df['Time Start'] = result_df['Time Stop'] - pd.to_timedelta('2:00:00')

# Format the 'mrn' column to have exactly 9 digits with leading zeros
result_df['mrn'] = result_df['mrn'].astype(str).str.zfill(9)

# Save the DataFrame as a new Excel file
result_df.to_excel('sample_events_MRN_2hrsBeforeHAD.xlsx', index=False)

print("DataFrame saved as 'sample_events_MRN_2hrsBeforeHAD.xlsx'")

#### 6 hours after hospital admission date.

In [ ]:
import pandas as pd

result_df = sample_events_MRN.copy()

# Convert 'Time Start' to datetime format
result_df['Time Start'] = pd.to_datetime(result_df['Time Start'])

# Create new 'Time Stop' as 'Time Start' plus 18 hours
result_df['Time Stop'] = result_df['Time Start'] + pd.to_timedelta('18:00:00')

# Convert 'Time Stop' to datetime format
result_df['Time Stop'] = pd.to_datetime(result_df['Time Stop'])

# Format the 'mrn' column to have exactly 9 digits with leading zeros
result_df['mrn'] = result_df['mrn'].astype(str).str.zfill(9)

# Save the DataFrame as a new Excel file
result_df.to_excel('sample_events_MRN_6hrsAfterHAD.xlsx', index=False)

print("DataFrame saved as 'sample_events_MRN_6hrsAfterHAD.xlsx'")

### Events Summary (MRN, PID, Time Start, Time Stop)

In [ ]:
import pandas as pd

# File paths
path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData"
file1 = f"{path}/user_event_check_tag_103(1011).csv"
file2 = f"{path}/EventList_Vitals_6hrsBefore_18hrsAfter(1011).csv"

# Load the data from the CSV files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Sort df1 dataframes by 'Patient ID'
df1_sorted = df1.sort_values(by='Patient ID').reset_index(drop=True)

# Check if 'Patient ID' columns are the same
if df1_sorted['Patient ID'].tolist() == df2['Patient ID'].tolist():
    print("Patient ID columns are exactly the same.")
else:
    print("Patient ID columns do not match. Please ensure the order of Patient ID is the same in both files.")
    # Optionally, handle this case as needed, such as raising an exception or logging the issue

# Concatenate 'MRN' column from df1_sorted to df2_sorted
df2['MRN'] = df1_sorted['MRN']

# Keep only the specified columns
columns_to_keep = ['MRN', 'Patient ID', 'Time Start', 'Time Stop']
filtered_df = df2[columns_to_keep]

# Save the filtered dataframe to a new CSV file
output_file = f"{path}/Add_EventList_Vitals_1011.csv"
filtered_df.to_csv(output_file, index=False)

print(f"Merged and filtered file saved to {output_file}")


In [ ]:
import pandas as pd
import glob

# Define the path and file pattern
path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData"
file_pattern = f"{path}/Add_EventList_Vitals_*.csv"

# Get a sorted list of file paths
file_paths = sorted(glob.glob(file_pattern), key=lambda x: int(x.split('_')[-1].split('.')[0]))

# Initialize a list to hold the dataframes
dataframes = []

# Load each file and append its dataframe to the list
for file in file_paths:
    df = pd.read_csv(file)
    dataframes.append(df)

# Concatenate all dataframes in the list
stacked_df = pd.concat(dataframes, ignore_index=True)

# Save the stacked dataframe to a new CSV file
output_file = f"{path}/Add_EventList_Vitals_all.csv"
stacked_df.to_csv(output_file, index=False)

print(f"Stacked file saved to {output_file}")


### Check data validation and ventilation type 

In [ ]:
# Add file names to the event list
import pandas as pd
import os
import re

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/RawData"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.csv"

# Read the main event list CSV file
event_list_df = pd.read_csv(event_list_path)

# Get a list of files in the RawData directory
files = [f for f in os.listdir(raw_data_path) if f.endswith('.csv')]

# Extract numeric part within parentheses from filenames and sort them
def extract_number(filename):
    match = re.search(r'\((\d+)\)', filename)
    return int(match.group(1)) if match else float('inf')

# Sort files based on the numeric value extracted from the filename
files_sorted = sorted(files, key=extract_number)

# Check if the number of files matches the number of rows in the event list dataframe
if len(files_sorted) != len(event_list_df):
    raise ValueError("The number of files in the RawData directory does not match the number of rows in the event list dataframe.")

# Create a 'FileName' column with the sorted file names from the RawData directory
event_list_df['FileName'] = files_sorted

# Save the updated dataframe
output_file = f"{event_list_path.replace('.csv', '_with_Filenames.csv')}"
event_list_df.to_csv(output_file, index=False)

print(f"Updated file with 'FileName' column saved to {output_file}")


In [ ]:
# Determine if the file is valid (not empty)
import pandas as pd
import os

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/RawData"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.csv"

# Read the main event list CSV file
event_list_df = pd.read_csv(event_list_path)

# Function to check if columns 5 to 19 are all empty in a CSV file
def is_file_empty(file_path):
    df = pd.read_csv(file_path)
    # Check columns 5 to 19 (zero-indexed, so columns 4 to 18)
    columns_to_check = df.columns[4:19]  # Assuming columns 5 to 19 are indexed from 4 to 18
    if not all(col in df.columns for col in columns_to_check):
        # If the expected columns are not present, consider the file as not valid
        return False
    
    # Check if all values in these columns are missing
    return df[columns_to_check].isnull().all().all()

# Create a dictionary to store the validity of each file
validity_dict = {}

# Process each file in the RawData directory
files = sorted(f for f in os.listdir(raw_data_path) if f.endswith('.csv'))

for file in files:
    file_path = os.path.join(raw_data_path, file)
    # Check if the file is empty based on columns 5 to 19
    is_valid = 0 if is_file_empty(file_path) else 1
    # Store the result in the dictionary
    validity_dict[file] = is_valid

# Add the 'is_valid' column to the event_list_df
event_list_df['is_valid'] = event_list_df['FileName'].map(validity_dict).fillna(0).astype(int)

# Save the updated dataframe
output_file = f"{event_list_path.replace('.csv', '_with_is_valid.csv')}"
event_list_df.to_csv(output_file, index=False)

print(f"Updated file with 'is_valid' column saved to {output_file}")



In [ ]:
# Check the ventilator type within valid files
import pandas as pd
import os

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/RawData"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.csv"

# Read the main event list CSV file
event_list_df = pd.read_csv(event_list_path)

# Function to determine the ventilator type based on specific columns
def determine_vent_type(df):
    # Define columns to check (5th to 19th columns, 0-indexed)
    columns_to_check = df.columns[4:19]
    
    # Ventilator types to check
    vent_keywords = ["CDGR - PIP", "AVEA - PIP", "SVU - PIP"]
    
    # Check for the presence of non-empty columns with specific keywords
    for keyword in vent_keywords:
        keyword_columns = [col for col in columns_to_check if keyword in col]
        for col in keyword_columns:
            if df[col].dropna().astype(str).str.strip().ne('').any():
                return keyword.split(' ')[0]  # Return the type part before the space
    
    return 'none'

# Update the Vent_Type column based on the file content
for index, row in event_list_df.iterrows():
    if row['is_valid'] == 1:
        file_name = row['FileName']
        file_path = os.path.join(raw_data_path, file_name)
        
        # Load the file
        df = pd.read_csv(file_path)
        
        # Determine the Vent_Type for the file
        vent_type = determine_vent_type(df)
        
        # Update the Vent_Type in the event_list_df
        event_list_df.at[index, 'Vent_Type'] = vent_type

# Save the updated event list to a new CSV file
output_file = event_list_path.replace('.csv', '_with_vent_type.csv')
event_list_df.to_csv(output_file, index=False)

print(f"Updated event list file saved to {output_file}")


In [ ]:
# Locate the 1st hour's Start Time and Stop Time, and the 12th hour's Start Time and Stop Time
import pandas as pd
import os
from datetime import datetime, timedelta

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/RawData"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.csv"

# Read the main event list CSV file
event_list_df = pd.read_csv(event_list_path)

# Function to find the time point where at least 5 out of 15 columns have data
def find_first_time_start(file_path):
    df = pd.read_csv(file_path)
    
    # Check columns 4 to 19 (0-based index 4 to 18)
    columns_to_check = df.columns[4:19]  
    
    # Loop through each row to find the time point
    for _, row in df.iterrows():
        # Check how many of the columns have non-empty data
        count_non_empty = sum(pd.notna(row[columns_to_check]))
        if count_non_empty >= 5:
            return row['Time']  # Assuming the column name for time is 'Time'
    
    return None

# Initialize the '1st_Time_Start' column with None
event_list_df['1st_Time_Start'] = None

# Loop through each row in event_list_df
for index, row in event_list_df.iterrows():
    if row['is_valid'] == 1:
        # Construct the file path
        file_name = row['FileName']
        file_path = os.path.join(raw_data_path, file_name)
        
        # Find the first time start for the file
        first_time_start = find_first_time_start(file_path)
        
        # Update the '1st_Time_Start' column
        event_list_df.at[index, '1st_Time_Start'] = first_time_start

# Convert time columns to datetime
event_list_df['1st_Time_Start'] = pd.to_datetime(event_list_df['1st_Time_Start'], errors='coerce')

# Add 60 minutes to '1st_Time_Start' to get '1st_Time_Stop'
event_list_df['1st_Time_Stop'] = event_list_df['1st_Time_Start'] + timedelta(minutes=60)

# Add 12 hours to '1st_Time_Start' to get '12th_Time_Start'
event_list_df['12th_Time_Start'] = event_list_df['1st_Time_Start'] + timedelta(hours=12)

# Add 60 minutes to '12th_Time_Start' to get '12th_Time_Stop'
event_list_df['12th_Time_Stop'] = event_list_df['12th_Time_Start'] + timedelta(minutes=60)

# Ensure the MRN column contains 9 digits
event_list_df['MRN'] = event_list_df['MRN'].astype(str).str.zfill(9)

# Save the updated event list to an Excel file
output_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.xlsx"
event_list_df.to_excel(output_file, index=False)

print(f"Updated event list file saved to {output_file}")


In [ ]:
# Check for duplicate rows based on the 1st_Time_Start column
import pandas as pd

# File path
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all.xlsx"

# Read the Excel file
event_list_df = pd.read_excel(file_path)

# Identify duplicates based on 'MRN' and '1st_Time_Start'
event_list_df['is_duplicated'] = event_list_df.duplicated(subset=['MRN', '1st_Time_Start'], keep=False)

# Save the updated DataFrame with the new column as an XLSX file
output_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/Add_EventList_Vitals_all_updated.xlsx"
event_list_df.to_excel(output_file_path, index=False)

print(f"Updated file saved successfully at {output_file_path}")


In [ ]:
# Check time interval if 1st_Time_Stop is out of range
import pandas as pd
from datetime import timedelta

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/RawData"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/0-Add_EventList_Vitals_all.xlsx"

# Read the main event list Excel file
event_list_df = pd.read_excel(event_list_path)

# Assuming '1st_Time_Stop' and 'Time_Stop' columns already exist in event_list_df
# Convert '1st_Time_Stop' and 'Time_Stop' columns to datetime
event_list_df['1st_Time_Stop'] = pd.to_datetime(event_list_df['1st_Time_Stop'], errors='coerce')
event_list_df['Time Stop'] = pd.to_datetime(event_list_df['Time Stop'], errors='coerce')

# Check if '1st_Time_Stop' is greater than 'Time_Stop'
event_list_df['is_greater'] = event_list_df['1st_Time_Stop'] > event_list_df['Time Stop']

# Ensure the MRN column contains 9 digits
event_list_df['MRN'] = event_list_df['MRN'].astype(str).str.zfill(9)

# Save the updated event list to a new Excel file
output_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/sample_events_MRN_6hrsBefore_and_18hrsAfterHAD/VitalData/0-Add_EventList_Vitals_all.xlsx"
event_list_df.to_excel(output_file, index=False)

print(f"Updated event list file saved to {output_file}")


In [ ]:
# Check the 
import pandas as pd

# File paths
file1 = "FOCI_screening_PARDS_20240124.xlsx"
file2 = "Add_EventList_Vitals_all.xlsx"
output_file_path = "Add_EventList_Vitals_all.xlsx"

# Reading the "DataSummary" sheet from the first Excel file
data_summary_df = pd.read_excel(file1, sheet_name="DataSummary")

# Reading the second Excel file
event_list_df = pd.read_excel(file2)

# Subsetting data_summary_df where "Duration" column equals "1st"
subset_data_summary_df = data_summary_df[data_summary_df["Duration"] == "1st"]

# Create copies of the subsets to avoid the SettingWithCopyWarning
subset_data_summary_df = subset_data_summary_df.copy()
event_list_df = event_list_df.copy()

# Convert "Time Start" and "1st_Time_Start" to datetime if they are not already in datetime format
subset_data_summary_df["Time Start"] = pd.to_datetime(subset_data_summary_df["Time Start"])
event_list_df["1st_Time_Start"] = pd.to_datetime(event_list_df["1st_Time_Start"])

# Extract the date and hour components
subset_data_summary_df["Date"] = subset_data_summary_df["Time Start"].dt.date
subset_data_summary_df["Hour"] = subset_data_summary_df["Time Start"].dt.hour

event_list_df["Date"] = event_list_df["1st_Time_Start"].dt.date
event_list_df["Hour"] = event_list_df["1st_Time_Start"].dt.hour

# Perform the merge to identify matches
matches_df = pd.merge(
    event_list_df,
    subset_data_summary_df[["Date", "Hour", "Patient ID", "MRN"]],
    on=["Date", "Hour", "Patient ID", "MRN"],
    how="left",
    indicator=True
)

# Add a new column "Match_Found" to indicate if there was a match
matches_df["Match_Found"] = matches_df["_merge"] == "both"

# Drop the helper column used for merging
matches_df.drop(columns=["_merge"], inplace=True)

# Save the updated event_list_df as an XLSX file
matches_df.to_excel(output_file_path, index=False)

print(f"File saved successfully at {output_file_path}")


In [ ]:
# Perform the merge to identify matches
matches_df = pd.merge(
    subset_data_summary_df,
    event_list_df[["Date", "Hour", "Patient ID", "MRN"]],
    on=["Date", "Hour", "Patient ID", "MRN"],
    how="left",
    indicator=True
)

# Add a new column "Match_Found" to indicate if there was a match
matches_df["Match_Found"] = matches_df["_merge"] == "both"

# Drop the helper column used for merging
matches_df.drop(columns=["_merge"], inplace=True)

# Save the updated event_list_df as an XLSX file
output_file_path = "OldDataSummary.xlsx"
matches_df.to_excel(output_file_path, index=False)

print(f"File saved successfully at {output_file_path}")

### Additional Dataset (Waveform)

In [ ]:
# 09/18/2024
import pandas as pd

# Path to your Excel file
file_path = '2-PARDS/WaveformData/Code/FOCI_screening_PARDS_20240124.xlsx'

# Read the specific sheet
df3 = pd.read_excel(file_path, sheet_name='DataSummary')

# Filter the dataframe where 'Duration' is '1st'
df3 = df3[df3['Duration'] == '1st']

# Rename the PatientID column to PID
df3.rename(columns={'Patient ID': 'PID'}, inplace=True)

# Change the data type of 'PID' and 'MRN' columns to int
df3['PID'] = df3['PID'].astype(int)
df3['MRN'] = df3['MRN'].astype(int)

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df3['MRN'] = df3['MRN'].astype(str).str.zfill(9)

# Display the first few rows of the dataframe
print(df3.head())

In [ ]:
import pandas as pd

# Read the csv files into DataFrames
mrns = pd.read_csv('2-PARDS/WaveformData/Code/user_event_check_tag_122.csv')

# Format the 'MRN' column to have exactly 9 digits with leading zeros
mrns['MRN'] = mrns['MRN'].astype(str).str.zfill(9)

# Rename the PatientID column to PID
mrns.rename(columns={'Patient ID': 'PID'}, inplace=True)

mrns_sub = mrns[mrns['is valid'] == 1]
mrns_sub

In [ ]:
import pandas as pd

# Convert 'Time Start' to datetime
mrns_sub['Time Start'] = pd.to_datetime(mrns_sub['Time Start'], errors='coerce')

# Drop rows with NaT in 'Time Start'
mrns_sub = mrns_sub.dropna(subset=['Time Start'])

# Create a new column with date and hour only (drop minute and seconds)
mrns_sub['Time Start DateHour'] = mrns_sub['Time Start'].dt.floor('H')

# Create a combined MRN + Time Start column
mrns_sub['MRN_TimeStart'] = mrns_sub['MRN'].astype(str) + '_' + mrns_sub['Time Start DateHour'].astype(str)

# Check for duplicates
duplicates = mrns_sub[mrns_sub.duplicated(subset=['MRN_TimeStart'], keep=False)]

# Print the duplicates if any
if not duplicates.empty:
    print(f"Found {len(duplicates)} duplicate rows in MRN + Time Start combinations:")
    print(duplicates)
else:
    print("No duplicates found in MRN + Time Start combinations.")


In [ ]:
import pandas as pd

# Convert 'Time Start' to datetime, and handle any invalid formats
df3['Time Start'] = pd.to_datetime(df3['Time Start'], errors='coerce')
mrns_sub['Time Start'] = pd.to_datetime(mrns_sub['Time Start'], errors='coerce')

# Drop rows where 'Time Start' could not be parsed
df3 = df3.dropna(subset=['Time Start'])
print(len(df3['MRN'].unique()))
mrns_sub = mrns_sub.dropna(subset=['Time Start'])
print(len(mrns_sub['MRN']))

# Create new columns with date and hour only (drop minute and seconds)
df3['Time Start DateHour'] = df3['Time Start'].dt.floor('H')
mrns_sub['Time Start DateHour'] = mrns_sub['Time Start'].dt.floor('H')

# Create combined MRN + Time Start columns in both dataframes
df3['MRN_TimeStart'] = df3['MRN'].astype(str) + '_' + df3['Time Start DateHour'].astype(str)
mrns_sub['MRN_TimeStart'] = mrns_sub['MRN'].astype(str) + '_' + mrns_sub['Time Start DateHour'].astype(str)

# Find how many MRN + Time Start combinations in df3 are also in mrns_sub
matches_count = df3['MRN_TimeStart'].isin(mrns_sub['MRN_TimeStart']).sum()

print(f"Number of MRN + Time Start combinations in df3 that are also in mrns_sub: {matches_count}")


#########################################################

### Comparison of Old (56) and New (282) Cohorts

#########################################################

In [35]:
# Read Old Cohort DataSummary data
import pandas as pd

# Path to your Excel file
file_path = "/home/liuwent/2-PARDS/WaveformData/Data/FOCI_screening_PARDS_20240919.xlsx"

# Read the specific sheet
df1 = pd.read_excel(file_path, sheet_name='DataSummary')

# Filter the dataframe where 'Duration' is '1st'
df1 = df1[df1['Duration'] == '1st']

# Rename the PatientID column to PID
df1.rename(columns={'Patient ID': 'PID'}, inplace=True)

# Change the data type of 'PID' and 'MRN' columns to int
df1['PID'] = df1['PID'].astype(int)
df1['MRN'] = df1['MRN'].astype(int)

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df1['MRN'] = df1['MRN'].astype(str).str.zfill(9)

# Display the first few rows of the dataframe
print(df1.shape)
print(df1.head())

(58, 11)
  Cohort Ventilator Duration  File Name #  Patient #  48hrs Raw File #    PID  \
0     58       CDGR      1st          1.0        1.0               3.0   2951   
1     58       CDGR      1st          2.0        2.0               7.0  12800   
2     58       CDGR      1st          3.0        3.0              11.0  12833   
3     58       CDGR      1st          4.0        4.0              15.0  20146   
4     58       CDGR      1st          5.0        9.0              36.0  65281   

         MRN          Time Start           Time Stop Note  
0  101336219 2023-10-04 17:31:00 2023-10-04 18:31:00  NaN  
1  101569175 2023-10-17 21:56:00 2023-10-17 22:56:00  NaN  
2  100765255 2023-10-30 15:54:00 2023-10-30 16:54:00  NaN  
3  101562844 2023-10-06 15:17:00 2023-10-06 16:17:00  NaN  
4  040667398 2023-11-07 00:00:00 2023-11-07 01:00:00  NaN  


In [36]:
# Read Old Cohort EHR data 
import pandas as pd

# Read the csv files into DataFrames
df2 = pd.read_csv('/home/liuwent/2-PARDS/WaveformData/Data/EHR_data.csv')

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df2['MRN'] = df2['MRN'].astype(str).str.zfill(9)

print(df2.shape)
print(df2.head())

(57, 14)
   Cohort Ventilator      PID        MRN  vari_date  vari_ards  \
0      58      AVEAA    76598  101532084  18-Dec-23          0   
1      58      AVEAA   132310  101675324   1-Nov-23          1   
2      58      AVEAA   606288  101149608  19-Dec-23          0   
3      58      AVEAA  1026178  101890492  18-Dec-23          1   
4      58       CDGR     2951  101336219   4-Oct-23          0   

   vari_age_months  vari_age_years  vari_weight  \
0              707        1.936986         6.08   
1             2018        5.528767         9.84   
2             1957        5.361644        13.56   
3             1005        2.753425         5.49   
4             2884        7.901370        27.35   

                                 Mode Category                 File Name  \
0  Pressure Control (or Volume Guarantee) Mode  a_58_AVEAA_4_1st (1).csv   
1  Pressure Control (or Volume Guarantee) Mode  a_58_AVEAA_4_1st (2).csv   
2  Pressure Control (or Volume Guarantee) Mode  a_58_AVEAA_

In [37]:
# Merge two dfs
# Rename the merged DataFrame
df_old = df2.merge(df1[['Cohort', 'Ventilator', 'MRN', 'PID', 'Time Start', 'Time Stop']],
                   on=['Cohort', 'Ventilator', 'MRN', 'PID'],
                   how='left')

# Change 'vari_sex' values: 1 -> 'F', 2 -> 'M'
df_old['vari_sex'] = df_old['vari_sex'].replace({1: 'F', 2: 'M'})

# Reorder the columns
df_old = df_old[['PID', 'MRN', 'vari_ards', 'vari_age_years', 'vari_sex', 'vari_weight', 
                 'Time Start', 'Time Stop', 'File Name', 'File Name 2', 'Cohort', 
                 'Ventilator', 'Mode Category']]

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Add a new column 'Cohort_Type' and assign 'old' to all rows
df_old['Cohort_Type'] = 'old'


print(df_old.shape)
print(df_old.head())


(57, 14)
       PID        MRN  vari_ards  vari_age_years vari_sex  vari_weight  \
0    76598  101532084          0        1.936986        M         6.08   
1   132310  101675324          1        5.528767        M         9.84   
2   606288  101149608          0        5.361644        F        13.56   
3  1026178  101890492          1        2.753425        M         5.49   
4     2951  101336219          0        7.901370        M        27.35   

           Time Start           Time Stop                 File Name  \
0 2023-12-18 20:14:00 2023-12-18 21:14:00  a_58_AVEAA_4_1st (1).csv   
1 2023-11-01 13:10:00 2023-11-01 14:10:00  a_58_AVEAA_4_1st (2).csv   
2 2023-12-19 21:50:00 2023-12-19 22:50:00  a_58_AVEAA_4_1st (3).csv   
3 2023-12-18 21:09:00 2023-12-18 22:09:00  a_58_AVEAA_4_1st (4).csv   
4 2023-10-04 17:31:00 2023-10-04 18:31:00  a_58_CDGR_39_1st (1).csv   

                  File Name 2 Cohort Ventilator  \
0  a_58_AVEAA_4_24hrs (1).csv     58      AVEAA   
1  a_58_AVEAA_4_2

In [39]:
# Save the updated event list to a new Excel file
output_file = "df_old.xlsx"
df_old.to_excel(output_file, index=False)


In [21]:
# Read New Cohort DataSummary data
import pandas as pd

# Path to your Excel file
file_path = "/home/liuwent/2-PARDS/WaveformData/Data/Add_EventList_Vitals_all.xlsx"

# Read the specific sheet
df3 = pd.read_excel(file_path)

# Filter the dataframe where 'is_valid' is 1
df3 = df3[df3['is_valid'] == 1]

# Rename the PatientID column to PID
df3.rename(columns={'Patient ID': 'PID'}, inplace=True)

# Change the data type of 'PID' and 'MRN' columns to int
df3['PID'] = df3['PID'].astype(int)
df3['MRN'] = df3['MRN'].astype(int)

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df3['MRN'] = df3['MRN'].astype(str).str.zfill(9)

# Display the first few rows of the dataframe
print(df3.shape)
print(df3.head())

(307, 17)
          MRN    PID       Time Start           Time Stop  \
6   100765255  12833  10/30/2023 5:18 2023-10-31 05:18:00   
18  038702968  31678   5/20/2024 5:29 2024-05-21 05:29:00   
22  100749983  41926  3/12/2024 14:08 2024-03-13 14:08:00   
23  100160037  45581  4/12/2024 20:10 2024-04-13 20:10:00   
24  100160037  45581   4/13/2024 3:06 2024-04-14 03:06:00   

                                FileName  is_valid Vent_Type  \
6    Vitals_6hrsBefore_18hrsAfter(7).csv         1      CDGR   
18  Vitals_6hrsBefore_18hrsAfter(19).csv         1      CDGR   
22  Vitals_6hrsBefore_18hrsAfter(23).csv         1      CDGR   
23  Vitals_6hrsBefore_18hrsAfter(24).csv         1      CDGR   
24  Vitals_6hrsBefore_18hrsAfter(25).csv         1      CDGR   

        1st_Time_Start       1st_Time_Stop     12th_Time_Start  \
6  2023-10-30 15:54:21 2023-10-30 16:54:21 2023-10-31 03:54:21   
18 2024-05-20 19:34:05 2024-05-20 20:34:05 2024-05-21 07:34:05   
22 2024-03-13 08:53:31 2024-03-13 09:53:

In [22]:
# Check for missing rows in '1st_Time_Start', '1st_Time_Stop', '12th_Time_Start', and '12th_Time_Stop'
missing_rows = df3[df3[['1st_Time_Start', '1st_Time_Stop', 
'12th_Time_Start', '12th_Time_Stop']].isnull().any(axis=1)]

print(len(missing_rows))


9


In [23]:
# Check for duplicated rows based on the combination of 'MRN', 'PID', and '1st_Time_Start'
duplicated_rows = df3[df3.duplicated(subset=['PID', '1st_Time_Start'], keep=False)]

print(len(duplicated_rows))


35


In [24]:
# Read New Cohort events data (307 version)
import pandas as pd

# Read the csv files into DataFrames
df4 = pd.read_csv('/home/liuwent/2-PARDS/WaveformData/Data/user_event_check_tag_122.csv')

# Rename the PatientID column to PID
df4.rename(columns={'Patient ID': 'PID'}, inplace=True)

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df4['MRN'] = df4['MRN'].astype(str).str.zfill(9)

print(df4.shape)
print(df4.head())

(307, 11)
         MRN    PID           Time Start            Time Stop  \
0  100765255  12833  2023-10-30 15:54:21  2023-10-30 16:54:21   
1  038702968  31678  2024-05-20 19:34:05  2024-05-20 20:34:05   
2  100749983  41926  2024-03-13 08:53:31  2024-03-13 09:53:31   
3  100160037  45581  2024-04-13 02:10:01  2024-04-13 03:10:01   
4  100160037  45581  2024-04-13 03:06:01  2024-04-13 04:06:01   

           Time Start UTC           Time Stop UTC self_check  id_check  \
0  2023-10-30 19:54:21+00  2023-10-30 20:54:21+00        NaN       NaN   
1  2024-05-20 23:34:05+00  2024-05-21 00:34:05+00        NaN       NaN   
2  2024-03-13 12:53:31+00  2024-03-13 13:53:31+00        NaN       NaN   
3  2024-04-13 06:10:01+00  2024-04-13 07:10:01+00        NaN       NaN   
4  2024-04-13 07:06:01+00  2024-04-13 08:06:01+00        NaN       NaN   

   event_check  future_check  is valid  
0          NaN           NaN         1  
1          NaN           NaN         1  
2          NaN           NaN   

In [25]:
# Check for missing rows in 'Time Start', 'Time Stop'
missing_rows = df4[df4[['Time Start', 'Time Stop']].isnull().any(axis=1)]

print(len(missing_rows))

9


In [26]:
# Check for duplicated rows based on the combination of 'MRN', 'PID', and 'Time Start'
duplicated_rows = df4[df4.duplicated(subset=['PID', 'Time Start'], keep=False)]

print(len(duplicated_rows))

35


In [27]:
# Read New Cohort events data (282 version)
import pandas as pd

# Read the csv files into DataFrames
df5 = pd.read_csv('/home/liuwent/2-PARDS/WaveformData/Data/Study23_Tag122_EventList.csv')

# Rename the PatientID column to PID
df5.rename(columns={'Patient ID': 'PID'}, inplace=True)

print(df5.shape)
print(df5.head())

(282, 11)
    Mark   PID  Tag ID study_name  study_id          tag_name  \
0  17232  1288     122  PARDS_LWT        23  Add_1st_Waveform   
1  17233  1288     122  PARDS_LWT        23  Add_1st_Waveform   
2  17259  1293     122  PARDS_LWT        23  Add_1st_Waveform   
3  17260  1293     122  PARDS_LWT        23  Add_1st_Waveform   
4  17234  1748     122  PARDS_LWT        23  Add_1st_Waveform   

           Time Start UTC           Time Stop UTC           Time Start  \
0  2023-11-01 12:36:01+00  2023-11-01 13:36:01+00  2023-11-01 08:36:01   
1  2023-12-01 16:47:45+00  2023-12-01 17:47:45+00  2023-12-01 11:47:45   
2  2023-08-01 13:03:01+00  2023-08-01 14:03:01+00  2023-08-01 09:03:01   
3  2023-09-01 08:26:00+00  2023-09-01 09:26:00+00  2023-09-01 04:26:00   
4  2023-09-24 00:34:46+00  2023-09-24 01:34:46+00  2023-09-23 20:34:46   

             Time Stop                                    Patient Hx Link  
0  2023-11-01 09:36:01  /apps/46b9d48d-e360-4c44-b0e0-bf284c46683f/pat...  
1 

In [28]:
# Check for duplicated rows based on the combination of 'MRN', 'PID', and 'Time Start'
duplicated_rows = df5[df5.duplicated(subset=['PID', 'Time Start'], keep=False)]

print(len(duplicated_rows))

0


In [29]:
# Read New Cohort DataSummary data
df_sub

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_sub['MRN'] = df_sub['MRN'].astype(str).str.zfill(9)

df_sub.shape


(1011, 9)

In [30]:
# Ensure df_sub has unique MRN values
df_sub_unique = df_sub[['MRN', 'Age', 'Weight']].drop_duplicates(subset='MRN')

# Merge df_sub's 'Age' and 'Weight' into df3 based on 'MRN'
df3 = df3.merge(df_sub_unique, on='MRN', how='left')

df3


,MRN,PID,Time Start,Time Stop,FileName,is_valid,Vent_Type,1st_Time_Start,1st_Time_Stop,12th_Time_Start,12th_Time_Stop,is_greater,Date,Hour,Match_Found,is_duplicated,Notes,Age,Weight
0,100765255,12833,10/30/2023 5:18,2023-10-31 05:18:00,Vitals_6hrsBefore_18hrsAfter(7).csv,1,CDGR,2023-10-30 15:54:21,2023-10-30 16:54:21,2023-10-31 03:54:21,2023-10-31 04:54:21,False,2023-10-30,15.0,True,False,NaN,7.295232,NaN
1,038702968,31678,5/20/2024 5:29,2024-05-21 05:29:00,Vitals_6hrsBefore_18hrsAfter(19).csv,1,CDGR,2024-05-20 19:34:05,2024-05-20 20:34:05,2024-05-21 07:34:05,2024-05-21 08:34:05,False,2024-05-20,19.0,False,False,NaN,16.162991,2691.38
2,100749983,41926,3/12/2024 14:08,2024-03-13 14:08:00,Vitals_6hrsBefore_18hrsAfter(23).csv,1,CDGR,2024-03-13 08:53:31,2024-03-13 09:53:31,2024-03-13 20:53:31,2024-03-13 21:53:31,False,2024-03-13,8.0,False,False,NaN,21.488328,1689.61
3,100160037,45581,4/12/2024 20:10,2024-04-13 20:10:00,Vitals_6hrsBefore_18hrsAfter(24).csv,1,CDGR,2024-04-13 02:10:01,2024-04-13 03:10:01,2024-04-13 14:10:01,2024-04-13 15:10:01,False,2024-04-13,2.0,False,False,NaN,11.077591,814.82
4,100160037,45581,4/13/2024 3:06,2024-04-14 03:06:00,Vitals_6hrsBefore_18hrsAfter(25).csv,1,CDGR,2024-04-13 03:06:01,2024-04-13 04:06:01,2024-04-13 15:06:01,2024-04-13 16:06:01,False,2024-04-13,3.0,False,False,NaN,11.077591,814.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,041432081,833391,7/27/2023 14:36,2023-07-28 14:36:00,Vitals_6hrsBefore_18hrsAfter(1003).csv,1,CDGR,2023-07-28 00:35:14,2023-07-28 01:35:14,2023-07-28 12:35:14,2023-07-28 13:35:14,False,2023-07-28,0.0,False,True,NaN,12.541707,NaN
303,041432081,833391,7/27/2023 16:49,2023-07-28 16:49:00,Vitals_6hrsBefore_18hrsAfter(1004).csv,1,CDGR,2023-07-28 00:35:14,2023-07-28 01:35:14,2023-07-28 12:35:14,2023-07-28 13:35:14,False,2023-07-28,0.0,False,True,NaN,12.541707,NaN
304,100087414,879606,9/4/2023 11:35,2023-09-05 11:35:00,Vitals_6hrsBefore_18hrsAfter(1006).csv,1,CDGR,2023-09-04 17:53:59,2023-09-04 18:53:59,2023-09-05 05:53:59,2023-09-05 06:53:59,False,2023-09-04,17.0,False,False,NaN,11.197078,2190.49
305,100437378,928546,10/15/2023 9:39,2023-10-16 09:39:00,Vitals_6hrsBefore_18hrsAfter(1008).csv,1,CDGR,2023-10-15 15:44:48,2023-10-15 16:44:48,2023-10-16 03:44:48,2023-10-16 04:44:48,False,2023-10-15,15.0,False,False,NaN,19.087343,2793.67


In [32]:
# # First, ensure 'Time_Start' and '1st_Time_Start' are in the same format (datetime if needed)
# df5['Time Start'] = pd.to_datetime(df5['Time Start'])
# print(df5.shape)
# df3['1st_Time_Start'] = pd.to_datetime(df3['1st_Time_Start'])
# print(df3.shape)

# # If you want to remove rows where specific columns have missing values:
# df3 = df3.dropna(subset=['PID', '1st_Time_Start'])
# # Remove duplicate rows based on specific columns (for example, 'PID' and '1st_Time_Start')
# df3 = df3.drop_duplicates(subset=['PID', '1st_Time_Start'])
# print(df3.shape)


# Merge df3's selected columns with df5 based on matching 'PID' and '1st_Time_Start'
df5 = df5.merge(df3[['PID', 'MRN', 'Age', 'Weight', '1st_Time_Start', '1st_Time_Stop', '12th_Time_Start', '12th_Time_Stop', 'FileName', 'Vent_Type']],
                left_on=['PID', 'Time Start'], right_on=['PID', '1st_Time_Start'], how='left')

df5.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 282 entries, 0 to 281
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Mark             282 non-null    int64         
 1   PID              282 non-null    int64         
 2   Tag ID           282 non-null    int64         
 3   study_name       282 non-null    object        
 4   study_id         282 non-null    int64         
 5   tag_name         282 non-null    object        
 6   Time Start UTC   282 non-null    object        
 7   Time Stop UTC    282 non-null    object        
 8   Time Start       282 non-null    datetime64[ns]
 9   Time Stop        282 non-null    object        
 10  Patient Hx Link  282 non-null    object        
 11  MRN              282 non-null    object        
 12  Age              282 non-null    float64       
 13  Weight           240 non-null    float64       
 14  1st_Time_Start   282 non-null    datetime6

In [33]:
# Rename and keep the relevant columns from df5
df5_renamed = df5.rename(columns={
    'PID': 'PID',
    'MRN': 'MRN',
    'Age': 'Age',
    'Weight': 'Weight',
    '1st_Time_Start': '1st_Time_Start',
    '1st_Time_Stop': '1st_Time_Stop',
    '12th_Time_Start': '12th_Time_Start',
    '12th_Time_Stop': '12th_Time_Stop',
    'FileName': 'FileName',
    'Vent_Type': 'Vent_Type'
})[['PID', 'MRN', 'Age', 'Weight', '1st_Time_Start', '1st_Time_Stop', '12th_Time_Start', '12th_Time_Stop', 'FileName', 'Vent_Type']]

# Add a new column 'Cohort_Type' and assign the value 'new' to all rows
df5_renamed['Cohort_Type'] = 'new'

# Display the result
df5_renamed.head()


,PID,MRN,Age,Weight,1st_Time_Start,1st_Time_Stop,12th_Time_Start,12th_Time_Stop,FileName,Vent_Type,Cohort_Type
0,1288,101607118,0.828285,437.04,2023-11-01 08:36:01,2023-11-01 09:36:01,2023-11-01 20:36:01,2023-11-01 21:36:01,Vitals_6hrsBefore_18hrsAfter(707).csv,CDGR,new
1,1288,101607118,0.828285,437.04,2023-12-01 11:47:45,2023-12-01 12:47:45,2023-12-01 23:47:45,2023-12-02 00:47:45,Vitals_6hrsBefore_18hrsAfter(708).csv,CDGR,new
2,1293,101666704,0.409113,NaN,2023-08-01 09:03:01,2023-08-01 10:03:01,2023-08-01 21:03:01,2023-08-01 22:03:01,Vitals_6hrsBefore_18hrsAfter(808).csv,CDGR,new
3,1293,101666704,0.409113,NaN,2023-09-01 04:26:00,2023-09-01 05:26:00,2023-09-01 16:26:00,2023-09-01 17:26:00,Vitals_6hrsBefore_18hrsAfter(809).csv,CDGR,new
4,1748,101643395,1.277937,444.45,2023-09-23 20:34:46,2023-09-23 21:34:46,2023-09-24 08:34:46,2023-09-24 09:34:46,Vitals_6hrsBefore_18hrsAfter(712).csv,CDGR,new


In [38]:
df_new = df5_renamed

df_new.shape


(282, 11)

In [40]:
# Save the updated event list to a new Excel file
output_file = "df_new.xlsx"
df_new.to_excel(output_file, index=False)


### Combine Old and New

In [11]:
import pandas as pd

# Define the file paths
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/"
df_old_file = file_path + "df_old.xlsx"
df_new_file = file_path + "df_new.xlsx"

# Read the Excel files into pandas DataFrames
df_old = pd.read_excel(df_old_file)
df_new = pd.read_excel(df_new_file)

# Display the first few rows of each DataFrame to confirm the data is loaded
print(df_old.shape)
print(df_new.shape)


(57, 14)
(282, 11)


In [12]:
# Convert 'Time Start' and '1st_Time_Start' to datetime and extract only date and hour
df_old['Time Start H'] = pd.to_datetime(df_old['Time Start']).dt.floor('H')  # Round to the nearest hour
df_new['1st_Time_Start_H'] = pd.to_datetime(df_new['1st_Time_Start']).dt.floor('H')

# Convert 'Time Start' and '1st_Time_Start' to datetime and extract only date and minute
df_old['Time Start M'] = pd.to_datetime(df_old['Time Start']).dt.floor('T')  # Round to the nearest minute
df_new['1st_Time_Start_M'] = pd.to_datetime(df_new['1st_Time_Start']).dt.floor('T')  # Round to the nearest minute

# Check for matching rows based on MRN and the date-hour combination
matching_rows_old = df_old[(df_old['MRN'].isin(df_new['MRN'])) & 
                           (df_old['Time Start H'].isin(df_new['1st_Time_Start_H']))].index

matching_rows_old = df_old[(df_old['MRN'].isin(df_new['MRN'])) & 
                           (df_old['Time Start M'].isin(df_new['1st_Time_Start_M']))].index

matching_rows_new = df_new[(df_new['MRN'].isin(df_old['MRN'])) & 
                           (df_new['1st_Time_Start_H'].isin(df_old['Time Start H']))].index

matching_rows_new = df_new[(df_new['MRN'].isin(df_old['MRN'])) & 
                           (df_new['1st_Time_Start_M'].isin(df_old['Time Start M']))].index

# Print the number of matching rows in both dataframes
print("Number of matching rows in df_old:", len(matching_rows_old))
print("Number of matching rows in df_new:", len(matching_rows_new))

Number of matching rows in df_old: 46
Number of matching rows in df_new: 46


In [15]:
# Identify rows in df_old that are not in df_new based on MRN and hour-level Time Start
non_matching_rows_old = df_old[~df_old.set_index(['MRN', 'Time Start H']).index.isin(
                                df_new.set_index(['MRN', '1st_Time_Start_H']).index)]

# Format the 'MRN' column to have exactly 9 digits with leading zeros
non_matching_rows_old['MRN'] = non_matching_rows_old['MRN'].astype(str).str.zfill(9)
                               
# Output file path for the non-matching rows
output_file = file_path + "non_matching_rows_old.xlsx"

# Save the non-matching rows to an Excel file
non_matching_rows_old.to_excel(output_file, index=False)

# Display the non-matching rows in df_old
print(non_matching_rows_old)

       PID        MRN  vari_ards  vari_age_years vari_sex  vari_weight  \
4     2951  101336219          0        7.901370        M        27.35   
5   133796  042033169          0       13.569863        F        17.92   
12  621010  101740029          1        2.534247        F         7.12   
17  841156  101817684          0        0.276712        M         2.22   
39   85956  101610786          0       11.493151        M        12.75   
41  132254  100752993          0        6.358904        F        10.98   
45  890448  101840209          0       13.873973        F        23.74   
47  906660  101847693          0        0.043836        M         2.77   
49   84948  041994058          1       16.021917        F        41.96   
54  457847  040125760          0       14.095890        M        16.37   
55  890374  100932870          1        5.345205        M         9.30   

            Time Start           Time Stop                   File Name  \
4  2023-10-04 17:31:00 2023-10-04 18:

/tmp/ipykernel_2158238/2946815792.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_matching_rows_old['MRN'] = non_matching_rows_old['MRN'].astype(str).str.zfill(9)


#################################################################################

In [9]:
# 282 new cohort 1st hour vital data orgnization
import os
import pandas as pd

# Define paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/RawData"
save_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/RawData_1st"

# Load the df_new_file
df_new_file = pd.read_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new.xlsx")  # Adjust the path

# Ensure time columns are in datetime format
df_new_file['1st_Time_Start'] = pd.to_datetime(df_new_file['1st_Time_Start'])
df_new_file['1st_Time_Stop'] = pd.to_datetime(df_new_file['1st_Time_Stop'])

# Create a new column for the file names
df_new_file['FileName_Vital_1st'] = ""

# Initialize counters for debugging
files_processed = 0
files_skipped = 0

# Track generated file names to identify duplicates
generated_file_names = {}

# Loop through each row in df_new_file
for index, row in df_new_file.iterrows():
    # Get file name from the FileName column
    file_name = row['FileName']
    file_path = os.path.join(raw_data_path, file_name)

    # If the file exists
    if os.path.exists(file_path):
        # Read the CSV file
        df = pd.read_csv(file_path)

        # Ensure the Time column is in datetime format
        df['Time'] = pd.to_datetime(df['Time'])

        # Filter the data based on the 1st_Time_Start and 1st_Time_Stop
        start_time = row['1st_Time_Start']
        stop_time = row['1st_Time_Stop']
        filtered_df = df[(df['Time'] >= start_time) & (df['Time'] <= stop_time)]

        # If data is found within the time range
        if not filtered_df.empty:
            # Generate the new file name using PID and 1st_Time_Start
            pid = row['PID']
            new_file_name = f"{pid}_{start_time.strftime('%Y%m%d_%H')}_Vital_1st.csv"
            
            # Track file names to identify duplicates
            if new_file_name in generated_file_names:
                print(f"Duplicate file name found: {new_file_name} for row {index}")
            else:
                generated_file_names[new_file_name] = file_path

            # Save the filtered data to the new path
            new_file_path = os.path.join(save_path, new_file_name)
            filtered_df.to_csv(new_file_path, index=False)

            # Update df_new_file with the new file name
            df_new_file.at[index, 'FileName_Vital_1st'] = new_file_name

            files_processed += 1
            print(f"Processed and saved: {new_file_name}")
        else:
            print(f"No data found for time range: {start_time} to {stop_time} in file: {file_name}")
            files_skipped += 1
    else:
        print(f"File not found: {file_name}")
        files_skipped += 1

# Debugging info
print(f"Total files processed: {files_processed}")
print(f"Total files skipped (due to missing files or no data in range): {files_skipped}")

# Check for missing files in df_new_file
missing_files = df_new_file[df_new_file['FileName_Vital_1st'] == ""]
if not missing_files.empty:
    print(f"Missing files or no data in time range for the following rows:\n{missing_files[['FileName', 'PID', '1st_Time_Start', '1st_Time_Stop']]}")
else:
    print("All files processed successfully.")

# Save the updated df_new_file
df_new_file.to_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v2.xlsx", index=False)  # Adjust the path


Processed and saved: 1288_20231101_08_Vital_1st.csv
Processed and saved: 1288_20231201_11_Vital_1st.csv
Processed and saved: 1293_20230801_09_Vital_1st.csv
Processed and saved: 1293_20230901_04_Vital_1st.csv
Processed and saved: 1748_20230923_20_Vital_1st.csv
Processed and saved: 1748_20240221_02_Vital_1st.csv
Processed and saved: 2294_20240520_17_Vital_1st.csv
Processed and saved: 8444_20231019_00_Vital_1st.csv
Processed and saved: 10825_20231001_09_Vital_1st.csv
Processed and saved: 12800_20231017_21_Vital_1st.csv
Processed and saved: 12833_20231030_15_Vital_1st.csv
Processed and saved: 14627_20240127_10_Vital_1st.csv
Processed and saved: 14627_20240201_03_Vital_1st.csv
Processed and saved: 20146_20231006_15_Vital_1st.csv
Processed and saved: 20146_20240508_16_Vital_1st.csv
Processed and saved: 21072_20240315_16_Vital_1st.csv
Processed and saved: 22443_20240619_22_Vital_1st.csv
Processed and saved: 22818_20240615_12_Vital_1st.csv
Processed and saved: 31678_20240520_19_Vital_1st.csv
P

In [10]:
import os
import pandas as pd

# Define paths
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v2.xlsx"

# Load the df_new_file
df_new_file = pd.read_excel(file_path)

# Check for duplicates in the 'FileName_Vital_1st' column
duplicate_files = df_new_file[df_new_file.duplicated(subset='FileName_Vital_1st', keep=False)]

if not duplicate_files.empty:
    print("Duplicate file names found in the 'FileName_Vital_1st' column:")
    print(duplicate_files[['FileName_Vital_1st', 'PID', '1st_Time_Start']])
else:
    print("No duplicate file names found in the 'FileName_Vital_1st' column.")

# # Optionally, save the duplicate file rows for review
# duplicate_files.to_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/duplicate_files.xlsx", index=False)


Duplicate file names found in the 'FileName_Vital_1st' column:
                    FileName_Vital_1st      PID      1st_Time_Start
232  1081885_20240211_09_Vital_1st.csv  1081885 2024-02-11 09:00:00
233  1081885_20240211_09_Vital_1st.csv  1081885 2024-02-11 09:25:00


In [4]:
# 282 new cohort 12th hour vital data orgnization
import os
import pandas as pd

# Define paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/RawData"
save_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/RawData_12th"

# Load the df_new_file
df_new_file = pd.read_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v2.xlsx")  # Adjust the path
# df_new_file.shape

# Ensure time columns are in datetime format
df_new_file['12th_Time_Start'] = pd.to_datetime(df_new_file['12th_Time_Start'])
df_new_file['12th_Time_Stop'] = pd.to_datetime(df_new_file['12th_Time_Stop'])

# Create new column for the file names
df_new_file['FileName_Vital_12th'] = ""

# Loop through each row in df_new_file
for index, row in df_new_file.iterrows():
    # Get file name from the FileName column
    file_name = row['FileName']
    file_path = os.path.join(raw_data_path, file_name)

    # If file exists
    if os.path.exists(file_path):
        # Read the CSV file
        df = pd.read_csv(file_path)

        # Ensure the Time column is in datetime format
        df['Time'] = pd.to_datetime(df['Time'])

        # Filter the data based on the 1st_Time_Start and 1st_Time_Stop
        start_time = row['12th_Time_Start']
        stop_time = row['12th_Time_Stop']
        filtered_df = df[(df['Time'] >= start_time) & (df['Time'] <= stop_time)]

        # If data is found within the time range
        if not filtered_df.empty:
            # Generate the new file name using PID and 1st_Time_Start
            pid = row['PID']
            new_file_name = f"{pid}_{start_time.strftime('%Y%m%d_%H')}_Vital_12th.csv"

            # Save the filtered data to the new path
            filtered_df.to_csv(os.path.join(save_path, new_file_name), index=False)

            # Update df_new_file with the new file name
            df_new_file.at[index, 'FileName_Vital_12th'] = new_file_name

# Save the updated df_new_file
df_new_file.to_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v3.xlsx", index=False)  # Adjust the path


In [7]:
# 282 new cohort 1st hour waveform data orgnization
import os
import shutil  # For copying the files
import pandas as pd
import re  # For regular expression to extract row number from file name

# Define the paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/RawData"
rename_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/RENAME"
event_list_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/Study23_Tag122_EventList.csv"

# Create the "RENAME" folder if it doesn't exist
os.makedirs(rename_folder, exist_ok=True)

# Load the event list CSV file
event_list_df = pd.read_csv(event_list_file)

# Ensure the columns 'Patient ID' and 'Time Start' are available in the CSV
# Convert 'Time Start' to datetime to format it properly
event_list_df['Time Start'] = pd.to_datetime(event_list_df['Time Start'])

# Add a column to indicate the row number (starting from 1 for better readability)
event_list_df['Row_Number'] = event_list_df.index + 1

# Loop through each file in the raw data directory
for file_name in os.listdir(raw_data_path):
    # Check if the file name contains "Row_" and "_Data", then extract the row number
    match = re.search(r"Row_(\d+)_Data", file_name)
    
    if match:
        file_row_number = int(match.group(1))  # Extract the row number from the file name
        
        # Check if this row number matches any row in the 'Row_Number' column
        matching_row = event_list_df[event_list_df['Row_Number'] == file_row_number]
        
        if not matching_row.empty:
            # Get Patient ID and formatted Time Start from the matching row
            patient_id = matching_row.iloc[0]['Patient ID']
            time_start = matching_row.iloc[0]['Time Start'].strftime('%Y%m%d_%H')  # Format date and hour
            
            # Define the new file name based on Patient ID and Time Start
            new_file_name = f"{patient_id}_{time_start}.csv"
            current_file_path = os.path.join(raw_data_path, file_name)
            new_file_path = os.path.join(rename_folder, new_file_name)
            
            # Copy the file to the RENAME folder with the new name
            shutil.copy(current_file_path, new_file_path)
            print(f"Copied and renamed: {file_name} -> {new_file_name}")
            

Copied and renamed: Event_Row_229_Data_zero_order_interpolation.csv -> 1071829_20240201_14.csv
Copied and renamed: Event_Row_169_Data_zero_order_interpolation.csv -> 883140_20230907_14.csv
Copied and renamed: Event_Row_250_Data_zero_order_interpolation.csv -> 1134621_20240418_17.csv
Copied and renamed: Event_Row_258_Data_zero_order_interpolation.csv -> 1154740_20240401_03.csv
Copied and renamed: Event_Row_66_Data_zero_order_interpolation.csv -> 153543_20240101_09.csv
Copied and renamed: Event_Row_244_Data_zero_order_interpolation.csv -> 1117361_20240306_03.csv
Copied and renamed: Event_Row_99_Data_zero_order_interpolation.csv -> 511622_20231116_01.csv
Copied and renamed: Event_Row_152_Data_zero_order_interpolation.csv -> 848841_20230815_06.csv
Copied and renamed: Event_Row_171_Data_zero_order_interpolation.csv -> 890448_20230913_12.csv
Copied and renamed: Event_Row_141_Data_zero_order_interpolation.csv -> 802274_20231001_09.csv
Copied and renamed: Event_Row_180_Data_zero_order_interpol

In [14]:
# 282 new cohort 1st hour waveform data orgnization (Cont'd)
import os
import pandas as pd

# Define paths
raw_data_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/RENAME"
save_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/RawData_1st"

# Load the df_new_file
df_new_file = pd.read_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v3.xlsx")  # Adjust the path

# Ensure time columns are in datetime format
df_new_file['1st_Time_Start'] = pd.to_datetime(df_new_file['1st_Time_Start'])
df_new_file['1st_Time_Stop'] = pd.to_datetime(df_new_file['1st_Time_Stop'])

# Create new column for the file names
df_new_file['FileName_Waveform_1st'] = ""

# Loop through each row in df_new_file
for index, row in df_new_file.iterrows():
    # Get the PID and start time for filtering
    pid = row['PID']
    start_time = row['1st_Time_Start']
    
    # Extract date and hour for filename comparison
    date_hour_str = start_time.strftime('%Y%m%d_%H')
    
    # Construct the expected filename format
    expected_file_name = f"{pid}_{date_hour_str}.csv"
    file_path = os.path.join(raw_data_path, expected_file_name)

    # Check if the file exists
    if os.path.exists(file_path):
        # Read the CSV file
        df = pd.read_csv(file_path)

        # Ensure the Time column is in datetime format
        df['Time'] = pd.to_datetime(df['Time'])

        # Filter the data based on the 1st_Time_Start and 1st_Time_Stop
        stop_time = row['1st_Time_Stop']
        filtered_df = df[(df['Time'] >= start_time) & (df['Time'] <= stop_time)]

        # If data is found within the time range
        if not filtered_df.empty:
            # Generate the new file name using PID and 1st_Time_Start
            new_file_name = f"{pid}_{start_time.strftime('%Y%m%d_%H')}_Waveform_1st.csv"

            # Save the filtered data to the new path
            filtered_df.to_csv(os.path.join(save_path, new_file_name), index=False)

            # Update df_new_file with the new file name
            df_new_file.at[index, 'FileName_Waveform_1st'] = new_file_name

# Save the updated df_new_file
df_new_file.to_excel("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new_v4.xlsx", index=False)  # Adjust the path

Rename Old (11):

In [12]:
# read the df_old dataframe
import pandas as pd

# Define the file path
file_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'

# Read the Excel file
df_old = pd.read_excel(file_path)

# Display the first few rows of the dataframe
print(df_old.head())

       PID        MRN  vari_ards  vari_age_years vari_sex  vari_weight  \
0    76598  101532084          0        1.936986        M         6.08   
1   132310  101675324          1        5.528767        M         9.84   
2   606288  101149608          0        5.361644        F        13.56   
3  1026178  101890492          1        2.753425        M         5.49   
4     2951  101336219          0        7.901370        M        27.35   

       1st_Time_Start       1st_Time_Stop     12th_Time_Start  \
0 2023-12-18 20:14:00 2023-12-18 21:14:00 2023-12-19 08:14:00   
1 2023-11-01 13:10:00 2023-11-01 14:10:00 2023-11-02 01:10:00   
2 2023-12-19 21:50:00 2023-12-19 22:50:00 2023-12-20 09:50:00   
3 2023-12-18 21:09:00 2023-12-18 22:09:00 2023-12-19 09:09:00   
4 2023-10-04 17:31:00 2023-10-04 18:31:00 2023-10-05 05:31:00   

       12th_Time_Stop                 File Name                 File Name 2  \
0 2023-12-19 09:14:00  a_58_AVEAA_4_1st (1).csv  a_58_AVEAA_4_24hrs (1).csv   
1 2023

In [13]:
# # Convert "Time Start" and "Time Stop" columns to datetime if they are not already
# df_old['Time Start'] = pd.to_datetime(df_old['Time Start'])
# df_old['Time Stop'] = pd.to_datetime(df_old['Time Stop'])

# # Create new columns "12th_Time_Start" and "12th_Time_Stop"
# df_old['12th_Time_Start'] = df_old['Time Start'] + pd.Timedelta(hours=12)
# df_old['12th_Time_Stop'] = df_old['Time Stop'] + pd.Timedelta(hours=12)

# # Rename "Time Start" to "1st_Time_Start" and "Time Stop" to "1st_Time_Stop"
# df_old.rename(columns={'Time Start': '1st_Time_Start', 'Time Stop': '1st_Time_Stop'}, inplace=True)

# # Rearrange the column order: move "12th_Time_Start" and "12th_Time_Stop" after "1st_Time_Stop"
# cols = list(df_old.columns)
# # Find the correct index to insert the new columns
# idx = cols.index('1st_Time_Stop') + 1
# # Rearrange the columns
# new_cols = cols[:idx] + ['12th_Time_Start', '12th_Time_Stop'] + cols[idx:-2]  # Rest of the columns except the newly added ones
# df_old = df_old[new_cols]

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Display the first few rows to verify the changes
print(df_old.head())

# Save the modified df_old as an Excel file
df_old.to_excel(file_path, index=False)

print(f"The modified df_old has been saved to: {file_path}")

       PID        MRN  vari_ards  vari_age_years vari_sex  vari_weight  \
0    76598  101532084          0        1.936986        M         6.08   
1   132310  101675324          1        5.528767        M         9.84   
2   606288  101149608          0        5.361644        F        13.56   
3  1026178  101890492          1        2.753425        M         5.49   
4     2951  101336219          0        7.901370        M        27.35   

       1st_Time_Start       1st_Time_Stop     12th_Time_Start  \
0 2023-12-18 20:14:00 2023-12-18 21:14:00 2023-12-19 08:14:00   
1 2023-11-01 13:10:00 2023-11-01 14:10:00 2023-11-02 01:10:00   
2 2023-12-19 21:50:00 2023-12-19 22:50:00 2023-12-20 09:50:00   
3 2023-12-18 21:09:00 2023-12-18 22:09:00 2023-12-19 09:09:00   
4 2023-10-04 17:31:00 2023-10-04 18:31:00 2023-10-05 05:31:00   

       12th_Time_Stop                 File Name                 File Name 2  \
0 2023-12-19 09:14:00  a_58_AVEAA_4_1st (1).csv  a_58_AVEAA_4_24hrs (1).csv   
1 2023

In [16]:
import pandas as pd

# Load df_old
file_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'
df_old = pd.read_excel(file_path)

# Replace "24hrs" with "12th" in the "File Name 2" column
df_old['File Name 2'] = df_old['File Name 2'].str.replace('24hrs', '12th')

# Save the modified df_old back to an Excel file
save_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'
df_old.to_excel(save_path, index=False)

print(f"The modified 'File Name 2' column has been saved to: {save_path}")


The modified 'File Name 2' column has been saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx


In [15]:
import os
import pandas as pd
import shutil

# Define the paths
old_vital_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/03_Old_Raw_Vital_11_1st/'
rename_folder = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/03_Old_Raw_Vital_11_1st_RENAME/'

# Create the rename folder if it doesn't exist
os.makedirs(rename_folder, exist_ok=True)

# Load df_old with "File Name", "PID", "1st_Time_Start", and "1st_Time_Stop" columns
df_old = pd.read_excel('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx')

# Initialize an empty list to store the new file names
file_name_vital_1st_list = []

# Loop through each row in df_old
for index, row in df_old.iterrows():
    file_name = row['File Name']
    pid = row['PID']
    time_start = pd.to_datetime(row['1st_Time_Start'])
    time_stop = pd.to_datetime(row['1st_Time_Stop'])
    
    # Define the source CSV file path
    source_file = os.path.join(old_vital_path, file_name)
    
    # Check if the source file exists
    if os.path.exists(source_file):
        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(source_file)
        
        # Convert the "Time" column to datetime format for filtering
        df['Time'] = pd.to_datetime(df['Time'])
        
        # Subset the data based on the "Time Start" and "Time Stop"
        subset_df = df[(df['Time'] >= time_start) & (df['Time'] <= time_stop)]
        
        # Define the new file name
        new_file_name = f"{pid}_{time_start.strftime('%Y%m%d_%H')}_Vital_1st.csv"
        new_file_path = os.path.join(rename_folder, new_file_name)
        
        # Save the subset data to the new file
        subset_df.to_csv(new_file_path, index=False)
        
        # Add the new file name to the list
        file_name_vital_1st_list.append(new_file_name)
        
        print(f"File {new_file_name} saved successfully.")
    else:
        # Append None if the file is not found
        file_name_vital_1st_list.append(None)
        print(f"File {file_name} not found.")

# Add the new file name column to df_old
df_old['FileName_Vital_1st'] = file_name_vital_1st_list

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Save the modified df_old as an Excel file
save_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'
df_old.to_excel(save_path, index=False)

print(f"The modified df_old with 'FileName_Vital_1st' column has been saved to: {save_path}")


File a_58_AVEAA_4_1st (1).csv not found.
File a_58_AVEAA_4_1st (2).csv not found.
File a_58_AVEAA_4_1st (3).csv not found.
File a_58_AVEAA_4_1st (4).csv not found.
File 2951_20231004_17_Vital_1st.csv saved successfully.
File 133796_20231112_12_Vital_1st.csv saved successfully.
File a_58_CDGR_39_1st (12).csv not found.
File a_58_CDGR_39_1st (13).csv not found.
File a_58_CDGR_39_1st (14).csv not found.
File a_58_CDGR_39_1st (15).csv not found.
File a_58_CDGR_39_1st (16).csv not found.
File a_58_CDGR_39_1st (17).csv not found.
File 621010_20231021_10_Vital_1st.csv saved successfully.
File a_58_CDGR_39_1st (19).csv not found.
File a_58_CDGR_39_1st (2).csv not found.
File a_58_CDGR_39_1st (20).csv not found.
File a_58_CDGR_39_1st (21).csv not found.
File 841156_20231120_17_Vital_1st.csv saved successfully.
File a_58_CDGR_39_1st (23).csv not found.
File a_58_CDGR_39_1st (24).csv not found.
File a_58_CDGR_39_1st (25).csv not found.
File a_58_CDGR_39_1st (26).csv not found.
File a_58_CDGR_39_1

In [17]:
# 11 old cohort 12th hour vital data orgnization
import os
import pandas as pd
import shutil

# Define the paths
old_vital_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/04_Old_Raw_Vital_11_12th/'
rename_folder = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/04_Old_Raw_Vital_11_12th_RENAME/'

# Create the rename folder if it doesn't exist
os.makedirs(rename_folder, exist_ok=True)

# Load df_old with "File Name", "PID", "1st_Time_Start", and "1st_Time_Stop" columns
df_old = pd.read_excel('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx')

# Initialize an empty list to store the new file names
file_name_vital_12th_list = []

# Loop through each row in df_old
for index, row in df_old.iterrows():
    file_name = row['File Name 2']
    pid = row['PID']
    time_start = pd.to_datetime(row['12th_Time_Start'])
    time_stop = pd.to_datetime(row['12th_Time_Stop'])
    
    # Define the source CSV file path
    source_file = os.path.join(old_vital_path, file_name)
    
    # Check if the source file exists
    if os.path.exists(source_file):
        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(source_file)
        
        # Convert the "Time" column to datetime format for filtering
        df['Time'] = pd.to_datetime(df['Time'])
        
        # Subset the data based on the "Time Start" and "Time Stop"
        subset_df = df[(df['Time'] >= time_start) & (df['Time'] <= time_stop)]
        
        # Define the new file name
        new_file_name = f"{pid}_{time_start.strftime('%Y%m%d_%H')}_Vital_12th.csv"
        new_file_path = os.path.join(rename_folder, new_file_name)
        
        # Save the subset data to the new file
        subset_df.to_csv(new_file_path, index=False)
        
        # Add the new file name to the list
        file_name_vital_12th_list.append(new_file_name)
        
        print(f"File {new_file_name} saved successfully.")
    else:
        # Append None if the file is not found
        file_name_vital_12th_list.append(None)
        print(f"File {file_name} not found.")

# Add the new file name column to df_old
df_old['FileName_Vital_12th'] = file_name_vital_12th_list

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Save the modified df_old as an Excel file
save_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'
df_old.to_excel(save_path, index=False)

print(f"The modified df_old with 'FileName_Vital_1st' column has been saved to: {save_path}")


File a_58_AVEAA_4_12th (1).csv not found.
File a_58_AVEAA_4_12th (2).csv not found.
File a_58_AVEAA_4_12th (3).csv not found.
File a_58_AVEAA_4_12th (4).csv not found.
File 2951_20231005_05_Vital_12th.csv saved successfully.
File 133796_20231113_00_Vital_12th.csv saved successfully.
File a_58_CDGR_39_12th (12).csv not found.
File a_58_CDGR_39_12th (13).csv not found.
File a_58_CDGR_39_12th (14).csv not found.
File a_58_CDGR_39_12th (15).csv not found.
File a_58_CDGR_39_12th (16).csv not found.
File a_58_CDGR_39_12th (17).csv not found.
File 621010_20231021_22_Vital_12th.csv saved successfully.
File a_58_CDGR_39_12th (19).csv not found.
File a_58_CDGR_39_12th (2).csv not found.
File a_58_CDGR_39_12th (20).csv not found.
File a_58_CDGR_39_12th (21).csv not found.
File 841156_20231121_05_Vital_12th.csv saved successfully.
File a_58_CDGR_39_12th (23).csv not found.
File a_58_CDGR_39_12th (24).csv not found.
File a_58_CDGR_39_12th (25).csv not found.
File a_58_CDGR_39_12th (26).csv not foun

In [18]:
# 11 old cohort 1st hour waveform data orgnization
import os
import pandas as pd
import shutil

# Define the paths
old_vital_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/02_Old_Raw_Waveform_11_1st/'
rename_folder = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/02_Old_Raw_Waveform_11_1st_RENAME/'

# Create the rename folder if it doesn't exist
os.makedirs(rename_folder, exist_ok=True)

# Load df_old with "File Name", "PID", "1st_Time_Start", and "1st_Time_Stop" columns
df_old = pd.read_excel('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx')

# Initialize an empty list to store the new file names
file_name_waveform_1st_list = []

# Loop through each row in df_old
for index, row in df_old.iterrows():
    file_name = row['File Name']
    pid = row['PID']
    time_start = pd.to_datetime(row['1st_Time_Start'])
    time_stop = pd.to_datetime(row['1st_Time_Stop'])
    
    # Define the source CSV file path
    source_file = os.path.join(old_vital_path, file_name)
    
    # Check if the source file exists
    if os.path.exists(source_file):
        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(source_file)
        
        # Convert the "Time" column to datetime format for filtering
        df['Time'] = pd.to_datetime(df['Time'])
        
        # Subset the data based on the "Time Start" and "Time Stop"
        subset_df = df[(df['Time'] >= time_start) & (df['Time'] <= time_stop)]
        
        # Define the new file name
        new_file_name = f"{pid}_{time_start.strftime('%Y%m%d_%H')}_Waveform_1st.csv"
        new_file_path = os.path.join(rename_folder, new_file_name)
        
        # Save the subset data to the new file
        subset_df.to_csv(new_file_path, index=False)
        
        # Add the new file name to the list
        file_name_waveform_1st_list.append(new_file_name)
        
        print(f"File {new_file_name} saved successfully.")
    else:
        # Append None if the file is not found
        file_name_waveform_1st_list.append(None)
        print(f"File {file_name} not found.")

# Add the new file name column to df_old
df_old['FileName_Waveform_1st'] = file_name_waveform_1st_list

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Save the modified df_old as an Excel file
save_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx'
df_old.to_excel(save_path, index=False)

print(f"The modified df_old with 'FileName_Vital_1st' column has been saved to: {save_path}")

File a_58_AVEAA_4_1st (1).csv not found.
File a_58_AVEAA_4_1st (2).csv not found.
File a_58_AVEAA_4_1st (3).csv not found.
File a_58_AVEAA_4_1st (4).csv not found.
File 2951_20231004_17_Waveform_1st.csv saved successfully.
File 133796_20231112_12_Waveform_1st.csv saved successfully.
File a_58_CDGR_39_1st (12).csv not found.
File a_58_CDGR_39_1st (13).csv not found.
File a_58_CDGR_39_1st (14).csv not found.
File a_58_CDGR_39_1st (15).csv not found.
File a_58_CDGR_39_1st (16).csv not found.
File a_58_CDGR_39_1st (17).csv not found.
File 621010_20231021_10_Waveform_1st.csv saved successfully.
File a_58_CDGR_39_1st (19).csv not found.
File a_58_CDGR_39_1st (2).csv not found.
File a_58_CDGR_39_1st (20).csv not found.
File a_58_CDGR_39_1st (21).csv not found.
File 841156_20231120_17_Waveform_1st.csv saved successfully.
File a_58_CDGR_39_1st (23).csv not found.
File a_58_CDGR_39_1st (24).csv not found.
File a_58_CDGR_39_1st (25).csv not found.
File a_58_CDGR_39_1st (26).csv not found.
File a_

Old (11) some of the individual file need to frequency padding

In [11]:
import pandas as pd

# Path to the original CSV file
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/457847_20230913_14_Vital_1st.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/890448_20230913_14_Vital_1st.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/906660_20230927_20_Vital_1st.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/84948_20230915_07_Vital_12th.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/457847_20230914_02_Vital_12th.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/890448_20230914_02_Vital_12th.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/906660_20230928_08_Vital_12th.csv"


# Read the CSV file
df = pd.read_csv(file_path)

# Drop columns that contain 'Flow' or 'iPress Wave' in the name
filtered_df = df.loc[:, ~df.columns.str.contains('Flow|iPress Wave', case=False)]

# Assuming each row represents 1/100th of a second (100 Hz), we'll resample to 1 Hz
# First, create a time index (if the original data doesn't have a time column)
filtered_df.index = pd.to_timedelta(range(len(filtered_df)), unit='s') / 100  # Creating artificial time index

# Resample the data to 1 Hz, taking the first value every second
resampled_df = filtered_df.resample('1S').first()

# Save the resampled file to a new path
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/457847_20230913_14_Vital_1st_resampled.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/890448_20230913_14_Vital_1st_resampled.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_1st/906660_20230927_20_Vital_1st_resampled.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/84948_20230915_07_Vital_12th.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/457847_20230914_02_Vital_12th.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/890448_20230914_02_Vital_12th.csv"
# new_file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/906660_20230928_08_Vital_12th.csv"
resampled_df.to_csv(new_file_path, index=False)

# Check row counts for confirmation
original_rows = len(df)
resampled_rows = len(resampled_df)
print(f"Original file had {original_rows} rows.")
print(f"Resampled file has {resampled_rows} rows.")
print(f"Resampled data saved to: {new_file_path}")



Original file had 352264 rows.
Resampled file has 3523 rows.
Resampled data saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/b_Old_Raw_Vital_11_12th/906660_20230928_08_Vital_12th.csv


Combine df_old and df_new

In [12]:
# Read df_old
import pandas as pd

# Path to the df_old.xlsx file
df_old_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx"

# Attempt to read the df_old.xlsx file
try:
    df_old = pd.read_excel(df_old_path)
    print("File read successfully.")
    # Display the first few rows of the dataframe to verify
    print(df_old.head())
except FileNotFoundError:
    print(f"File not found at: {df_old_path}. Please check the path.")


File read successfully.
       PID        MRN  vari_ards  vari_age_years vari_sex  vari_weight  \
0    76598  101532084          0        1.936986        M         6.08   
1   132310  101675324          1        5.528767        M         9.84   
2   606288  101149608          0        5.361644        F        13.56   
3  1026178  101890492          1        2.753425        M         5.49   
4     2951  101336219          0        7.901370        M        27.35   

       1st_Time_Start       1st_Time_Stop     12th_Time_Start  \
0 2023-12-18 20:14:00 2023-12-18 21:14:00 2023-12-19 08:14:00   
1 2023-11-01 13:10:00 2023-11-01 14:10:00 2023-11-02 01:10:00   
2 2023-12-19 21:50:00 2023-12-19 22:50:00 2023-12-20 09:50:00   
3 2023-12-18 21:09:00 2023-12-18 22:09:00 2023-12-19 09:09:00   
4 2023-10-04 17:31:00 2023-10-04 18:31:00 2023-10-05 05:31:00   

       12th_Time_Stop                 File Name                File Name 2  \
0 2023-12-19 09:14:00  a_58_AVEAA_4_1st (1).csv  a_58_AVEAA_4_

In [18]:
import pandas as pd

# Path to the df_old.xlsx file
df_old_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx"

# Read the df_old.xlsx file
df_old = pd.read_excel(df_old_path)

# Define the columns to be renamed
df_old.rename(columns={
    'vari_ards': 'ADRS',
    'vari_age_years': 'Age',
    'vari_sex': 'Sex',
    'vari_weight': 'Weight',
    'File Name': 'FileName',
    'File Name 2': 'FileName2',
    'Ventilator': 'Vent_Type',
    'Mode Category': 'Mode_Cat'
}, inplace=True)

# Display the first few rows to verify the changes
print(df_old.head())

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_old['MRN'] = df_old['MRN'].astype(str).str.zfill(9)

# Optional: Save the updated df_old file back to the same directory
df_old.to_excel(df_old_path, index=False)
print(f"Updated df_new saved to: {df_old_path}")

Updated df_new saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx


In [14]:
# Read df_new
import pandas as pd

# Path to the df_new.xlsx file
df_new_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new.xlsx"

# Attempt to read the df_new.xlsx file
try:
    df_new = pd.read_excel(df_new_path)
    print("File read successfully.")
    # Display the first few rows of the dataframe to verify
    print(df_new.head())
except FileNotFoundError:
    print(f"File not found at: {df_new_path}. Please check the path.")

File read successfully.
    PID        MRN       Age  Weight      1st_Time_Start       1st_Time_Stop  \
0  1288  101607118  0.828285  437.04 2023-11-01 08:36:01 2023-11-01 09:36:01   
1  1288  101607118  0.828285  437.04 2023-12-01 11:47:45 2023-12-01 12:47:45   
2  1293  101666704  0.409113     NaN 2023-08-01 09:03:01 2023-08-01 10:03:01   
3  1293  101666704  0.409113     NaN 2023-09-01 04:26:00 2023-09-01 05:26:00   
4  1748  101643395  1.277937  444.45 2023-09-23 20:34:46 2023-09-23 21:34:46   

      12th_Time_Start      12th_Time_Stop  \
0 2023-11-01 20:36:01 2023-11-01 21:36:01   
1 2023-12-01 23:47:45 2023-12-02 00:47:45   
2 2023-08-01 21:03:01 2023-08-01 22:03:01   
3 2023-09-01 16:26:00 2023-09-01 17:26:00   
4 2023-09-24 08:34:46 2023-09-24 09:34:46   

                                FileName Vent_Type Cohort_Type  \
0  Vitals_6hrsBefore_18hrsAfter(707).csv      CDGR         new   
1  Vitals_6hrsBefore_18hrsAfter(708).csv      CDGR         new   
2  Vitals_6hrsBefore_18hrs

In [19]:
# Replace "AVEA" with "AVEAA" in the 'Vent_Type' column
df_new['Vent_Type'] = df_new['Vent_Type'].replace('AVEA', 'AVEAA')

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_new['MRN'] = df_new['MRN'].astype(str).str.zfill(9)

# Display the first few rows to verify the changes
print(df_new.head())

# Optional: Save the updated df_new file back to the same directory
df_new.to_excel(df_new_path, index=False)
print(f"Updated df_new saved to: {df_new_path}")

    PID        MRN       Age  Weight      1st_Time_Start       1st_Time_Stop  \
0  1288  101607118  0.828285  437.04 2023-11-01 08:36:01 2023-11-01 09:36:01   
1  1288  101607118  0.828285  437.04 2023-12-01 11:47:45 2023-12-01 12:47:45   
2  1293  101666704  0.409113     NaN 2023-08-01 09:03:01 2023-08-01 10:03:01   
3  1293  101666704  0.409113     NaN 2023-09-01 04:26:00 2023-09-01 05:26:00   
4  1748  101643395  1.277937  444.45 2023-09-23 20:34:46 2023-09-23 21:34:46   

      12th_Time_Start      12th_Time_Stop  \
0 2023-11-01 20:36:01 2023-11-01 21:36:01   
1 2023-12-01 23:47:45 2023-12-02 00:47:45   
2 2023-08-01 21:03:01 2023-08-01 22:03:01   
3 2023-09-01 16:26:00 2023-09-01 17:26:00   
4 2023-09-24 08:34:46 2023-09-24 09:34:46   

                                FileName Vent_Type Cohort_Type  \
0  Vitals_6hrsBefore_18hrsAfter(707).csv      CDGR         new   
1  Vitals_6hrsBefore_18hrsAfter(708).csv      CDGR         new   
2  Vitals_6hrsBefore_18hrsAfter(808).csv      CDGR

In [21]:
import pandas as pd

# Paths to df_old and df_new files
df_old_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_old.xlsx"
df_new_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_new.xlsx"

# Read the Excel files
df_old = pd.read_excel(df_old_path)
df_new = pd.read_excel(df_new_path)

# Concatenate df_new (282 rows) followed by df_old (57 rows)
df_combined = pd.concat([df_new, df_old], axis=0, ignore_index=True, sort=False)

# Format the 'MRN' column to have exactly 9 digits with leading zeros
df_combined['MRN'] = df_combined['MRN'].astype(str).str.zfill(9)

# Display the first few rows to verify the concatenation and columns
print(df_combined.head())
print(df_combined.columns)

# Optional: Save the combined DataFrame to a new Excel file
output_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
df_combined.to_excel(output_path, index=False)
print(f"Concatenated DataFrame saved to: {output_path}")

    PID        MRN       Age  Weight      1st_Time_Start       1st_Time_Stop  \
0  1288  101607118  0.828285  437.04 2023-11-01 08:36:01 2023-11-01 09:36:01   
1  1288  101607118  0.828285  437.04 2023-12-01 11:47:45 2023-12-01 12:47:45   
2  1293  101666704  0.409113     NaN 2023-08-01 09:03:01 2023-08-01 10:03:01   
3  1293  101666704  0.409113     NaN 2023-09-01 04:26:00 2023-09-01 05:26:00   
4  1748  101643395  1.277937  444.45 2023-09-23 20:34:46 2023-09-23 21:34:46   

      12th_Time_Start      12th_Time_Stop  \
0 2023-11-01 20:36:01 2023-11-01 21:36:01   
1 2023-12-01 23:47:45 2023-12-02 00:47:45   
2 2023-08-01 21:03:01 2023-08-01 22:03:01   
3 2023-09-01 16:26:00 2023-09-01 17:26:00   
4 2023-09-24 08:34:46 2023-09-24 09:34:46   

                                FileName Vent_Type Cohort_Type  \
0  Vitals_6hrsBefore_18hrsAfter(707).csv      CDGR         new   
1  Vitals_6hrsBefore_18hrsAfter(708).csv      CDGR         new   
2  Vitals_6hrsBefore_18hrsAfter(808).csv      CDGR

checks for duplicate rows based on the combination of PID, MRN, and the date part of 1st_Time_Start

In [5]:
import pandas as pd

# Define the path to the Excel file
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"

# Read the Excel file into a DataFrame
df_combined = pd.read_excel(file_path)

# Subset the DataFrame where 'FileName_Vital_1st' is not empty
df_subset = df_combined[df_combined['FileName_Vital_1st'].notna()].copy()

# Convert '1st_Time_Start' to datetime and extract the date part using .loc to avoid the warning
df_subset.loc[:, '1st_Time_Start'] = pd.to_datetime(df_subset['1st_Time_Start'], errors='coerce')
df_subset.loc[:, '1st_Time_Start_Date'] = df_subset['1st_Time_Start'].dt.date

# Check for duplicates based on PID, MRN, and '1st_Time_Start_Date'
duplicates = df_subset[df_subset.duplicated(subset=['PID', 'MRN', '1st_Time_Start_Date'], keep=False)]

# Print the duplicate rows
print("Duplicate rows based on PID, MRN, and 1st_Time_Start date (when FileName_Vital_1st is not empty):")
print(duplicates)


Duplicate rows based on PID, MRN, and 1st_Time_Start date (when FileName_Vital_1st is not empty):
         PID        MRN        Age  Weight      1st_Time_Start  \
20     45581  100160037  11.077591  814.82 2024-04-13 02:10:01   
21     45581  100160037  11.077591  814.82 2024-04-13 03:06:01   
45     86566  100572143   9.312898  670.20 2024-01-01 06:11:01   
46     86566  100572143   9.312898  670.20 2024-01-01 08:52:01   
62    153543  100574112   8.518260     NaN 2023-12-01 01:15:01   
63    153543  100574112   8.518260     NaN 2023-12-01 08:38:01   
90    354894  101001701   5.490167  656.09 2024-04-27 11:43:27   
91    354894  101001701   5.490167  656.09 2024-04-27 12:16:01   
169   890374  100932870   5.339360  691.36 2023-09-13 05:46:55   
170   890448  101840209  13.923055     NaN 2023-09-13 12:59:54   
232  1081885  101916512   1.078685  426.81 2024-02-11 09:00:00   
233  1081885  101916512   1.078685  426.81 2024-02-11 09:25:00   
250  1134641  101935451  11.653405     NaN 2